<a href="https://colab.research.google.com/github/kshitij-nandanwar/OT-LAB/blob/main/OT_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Northwest Corner Method

In [ ]:
import numpy as np

cost = np.array([
    [2, 3, 1],
    [5, 4, 8],
    [5, 6, 8]
])

supply = [30, 40, 20]
demand = [20, 30, 40]

allocation = np.zeros((3,3), dtype=int)

i = 0
j = 0

while i < len(supply) and j < len(demand):

    x = min(supply[i], demand[j])

    allocation[i][j] = x

    supply[i] -= x
    demand[j] -= x

    if supply[i] == 0:
        i += 1
    elif demand[j] == 0:
        j += 1

print("Allocation Matrix:")
print(allocation)

total_cost = np.sum(allocation * cost)

print("\nTotal Transportation Cost =", total_cost)


Allocation Matrix:
[[20 10  0]
 [ 0 20 20]
 [ 0  0 20]]

Total Transportation Cost = 470


### 2. Least Cost Method (LCM)

In [ ]:
import numpy as np

def least_cost_method(cost_matrix, supply, demand):
    cost = cost_matrix.copy().astype(float)
    s = list(supply).copy()
    d = list(demand).copy()
    m, n = cost.shape
    allocation = np.zeros((m, n), dtype=int)

    while sum(s) > 0 and sum(d) > 0:
        min_val = np.inf
        min_i, min_j = -1, -1
        for i in range(m):
            if s[i] == 0:
                continue
            for j in range(n):
                if d[j] == 0:
                    continue
                if cost[i][j] < min_val:
                    min_val = cost[i][j]
                    min_i, min_j = i, j

        x = min(s[min_i], d[min_j])
        allocation[min_i][min_j] = x
        s[min_i] -= x
        d[min_j] -= x

    return allocation

costs = np.array([
    [2, 3, 1],
    [5, 4, 8],
    [5, 6, 8]
])
supply = [30, 40, 20]
demand = [20, 30, 40]

lcm_allocation = least_cost_method(costs, supply, demand)
lcm_cost = np.sum(lcm_allocation * costs)

print("Least Cost Method Allocation Matrix:")
print(lcm_allocation)
print("Total Transportation Cost (LCM) =", lcm_cost)

Least Cost Method Allocation Matrix:
[[ 0  0 30]
 [10 30  0]
 [10  0 10]]
Total Transportation Cost (LCM) = 330


### 3. Vogel’s Approximation Method (VAM)

In [ ]:
def vogel_approximation_method(cost_matrix, supply, demand):
    cost = cost_matrix.copy().astype(float)
    s = list(supply).copy()
    d = list(demand).copy()
    m, n = cost.shape
    allocation = np.zeros((m, n), dtype=int)

    while sum(s) > 0 and sum(d) > 0:
        row_penalties = []
        col_penalties = []

        for i in range(m):
            if s[i] == 0:
                row_penalties.append(-1)
                continue
            active_row_costs = [cost[i][j] for j in range(n) if d[j] > 0]
            if len(active_row_costs) > 1:
                sorted_costs = sorted(active_row_costs)
                row_penalties.append(sorted_costs[1] - sorted_costs[0])
            elif len(active_row_costs) == 1:
                row_penalties.append(active_row_costs[0])
            else:
                row_penalties.append(-1)

        for j in range(n):
            if d[j] == 0:
                col_penalties.append(-1)
                continue
            active_col_costs = [cost[i][j] for i in range(m) if s[i] > 0]
            if len(active_col_costs) > 1:
                sorted_costs = sorted(active_col_costs)
                col_penalties.append(sorted_costs[1] - sorted_costs[0])
            elif len(active_col_costs) == 1:
                col_penalties.append(active_col_costs[0])
            else:
                col_penalties.append(-1)

        max_row_pen = max(row_penalties)
        max_col_pen = max(col_penalties)

        if max_row_pen >= max_col_pen:
            row_idx = row_penalties.index(max_row_pen)
            col_idx = -1
            min_val = np.inf
            for j in range(n):
                if d[j] > 0 and cost[row_idx][j] < min_val:
                    min_val = cost[row_idx][j]
                    col_idx = j
        else:
            col_idx = col_penalties.index(max_col_pen)
            row_idx = -1
            min_val = np.inf
            for i in range(m):
                if s[i] > 0 and cost[i][col_idx] < min_val:
                    min_val = cost[i][col_idx]
                    row_idx = i

        x = min(s[row_idx], d[col_idx])
        allocation[row_idx][col_idx] = x
        s[row_idx] -= x
        d[col_idx] -= x

    return allocation

vam_allocation = vogel_approximation_method(costs, supply, demand)
vam_cost = np.sum(vam_allocation * costs)

print("Vogel's Approximation Method Allocation Matrix:")
print(vam_allocation)
print("Total Transportation Cost (VAM) =", vam_cost)

Vogel's Approximation Method Allocation Matrix:
[[ 0  0 30]
 [10 30  0]
 [10  0 10]]
Total Transportation Cost (VAM) = 330


### 4. Stepping Stone Method

In [ ]:
def get_u_v(costs, allocation):
    m, n = costs.shape
    u = [None] * m
    v = [None] * n
    u[0] = 0

    while None in u or None in v:
        for i in range(m):
            for j in range(n):
                if allocation[i][j] > 0:
                    if u[i] is not None and v[j] is None:
                        v[j] = costs[i][j] - u[i]
                    elif v[j] is not None and u[i] is None:
                        u[i] = costs[i][j] - v[j]
    return u, v

def get_opportunity_costs(costs, allocation, u, v):
    m, n = costs.shape
    opportunity_costs = np.zeros((m, n))
    for i in range(m):
        for j in range(n):
            if allocation[i][j] == 0:
                opportunity_costs[i][j] = costs[i][j] - (u[i] + v[j])
    return opportunity_costs

u, v = get_u_v(costs, vam_allocation)
opp_costs = get_opportunity_costs(costs, vam_allocation, u, v)

print("Optimality Evaluation (Opportunity Costs / Net Evaluations for empty cells):")
print(opp_costs)

if np.all(opp_costs >= 0):
    print("\nSince all opportunity costs are non-negative, the VAM allocation is optimal.")
else:
    print("\nImprovement is possible using Stepping Stone closed path.")

Optimality Evaluation (Opportunity Costs / Net Evaluations for empty cells):
[[4. 6. 0.]
 [0. 0. 0.]
 [0. 2. 0.]]

Since all opportunity costs are non-negative, the VAM allocation is optimal.
